# jax-compress

Made by Byron Knoll. GitHub repository: https://github.com/byronknoll/jax-compress

### Description
This project started as a JAX/Flax port of [tensorflow-compress](https://github.com/byronknoll/tensorflow-compress).

jax-compress performs lossless data compression using neural networks. It can run on TPUs and GPUs with a large batch size to achieve substantial speed improvements. It is designed for Google Colab, making it easy to run directly through a web browser. You can select a file, perform compression (or decompression), and download the resulting file.

The neural network is trained from scratch during both compression and decompression, meaning the model weights do not need to be stored in the compressed file. Arithmetic coding is used to encode the model predictions.

Feel free to contact me at byron@byronknoll.com if you have any questions.

### Instructions

**Basic Usage:** Configure all the fields in the "Parameters" section and select `Runtime -> Run All`.

**Advanced Usage:** Save a copy of this notebook to your own Google Drive and modify the code as needed.

### Related Projects
*   [tensorflow-compress](https://github.com/byronknoll/tensorflow-compress)
*   [NNCP](https://bellard.org/nncp/)
*   [lstm-compress](https://github.com/byronknoll/lstm-compress)
*   [cmix](http://www.byronknoll.com/cmix.html)

### Benchmarks
These benchmarks were performed using jax-compress v1 with the default parameter settings on a v6e-1 TPU. Compression time and decompression time are approximately the same.

*   **enwik8:** Compressed to 15,505,441 bytes in 13,707.98 seconds.
*   **enwik9:** Compressed to 113,393,442 bytes in 110,013.19 seconds (Dictionary size: 80,040 bytes). 
    * The preprocessed enwik9 file was split into two parts. 
    * The "checkpoint" option was used to save and load model weights between processing each part. 
    * Article ordering preprocessing (from [fx2-cmix](https://github.com/kaitz/fx2-cmix)) was used:

```bash
cd enwik9-preproc/
make
./enwik9-preproc c enwik9
# After decompression:
./enwik9-preproc d final.dat
```

  * enwik9 decompressor size is 60,872 bytes. It is a zip file which contains:
      * jax_compress notebook
      * enwik9 article reordering code. This doesn't include the new_article_order, which is only needed for compression.
      * NNCP preprocessor code
      * dictionary for enwik9


See the [Large Text Compression Benchmark](http://mattmahoney.net/dc/text.html) for more information about the test files and a comparison with other compression programs.

### Versions
* **v1** - Released March 15, 2026. Changes from tensorflow-compress:
  * **Retraining:** Similar to NNCP, the model is periodically retrained using previously processed data.
  * Fixed a bug in the NNCP preprocessor.

## Parameters

In [ ]:
#@title Parameters
#@markdown ---
batch_size = 128 #@param {type:"integer"}
#@markdown _Splits the file into N batches to process them in parallel. Increasing this value improves speed but may negatively affect the compression rate. For optimal speed on certain GPUs, set this to a multiple of 8._

#@markdown ---
seq_length =  15 #@param {type:"integer"}
#@markdown _Determines the horizon for backpropagation through time. Lowering this value improves speed but can reduce the compression rate._

#@markdown ---
rnn_units =  1400 #@param {type:"integer"}
#@markdown _The number of units to use within each LSTM layer. Lowering this improves speed and reduces memory usage, but can worsen the compression rate. Make this a multiple of 8 to improve speed on certain GPUs._

#@markdown ---
num_layers = 8 #@param {type:"integer"}
#@markdown _The number of LSTM layers to use. Lowering this value improves speed at the cost of the compression rate._

#@markdown ---
embedding_size=512 #@param {type:"integer"}
#@markdown _The size of the embedding layer._

#@markdown ---
ensemble_size = 1 #@param {type:"integer"}
#@markdown _The number of LSTM models to use in the ensemble._

#@markdown ---
learning_rate_schedule = "0:0.0005 200000:0.0002" #@param {type:"string"}
#@markdown _The learning rate schedule formatted as "step:value". Values are linearly interpolated between points. Special handling is applied for the first and last points: from the start of the file to the first point, the value remains constant. Similarly, it stays constant from the last data point to the end of the file._

#@markdown ---
adam_b1 = 0.0 #@param {type:"number"}
#@markdown _The Beta1 parameter for the Adam optimizer._

#@markdown ---
adam_b2 = 0.9999 #@param {type:"number"}
#@markdown _The Beta2 parameter for the Adam optimizer._

#@markdown ---
adam_eps = 1e-12 #@param {type:"number"}
#@markdown _The Epsilon parameter for the Adam optimizer._

#@markdown ---
mode = 'compress' #@param ["compress", "decompress", "both", "preprocess_only"]
#@markdown _Select the execution mode. "preprocess_only" will exclusively run preprocessing and skip the compression step._

#@markdown ---
preprocess = 'nncp' #@param ["cmix", "nncp", "nncp-done", "none"]
#@markdown _Select the preprocessor. NNCP performs better on enwik8/enwik9. It is slower because it builds a custom dictionary, whereas cmix uses a pretrained one. "nncp-done" is for files already preprocessed by NNCP (the dictionary must be included)._

#@markdown ---
n_words = 8192 #@param {type:"integer"}
#@markdown _For NNCP preprocessor only: the approximate maximum dictionary size. Recommended values: 8192 for enwik8 and enwik9._

#@markdown ---
min_freq = 64 #@param {type:"integer"}
#@markdown _For NNCP preprocessor only: the minimum frequency required for selected words. Recommended values: 64 for enwik8, 512 for enwik9._

#@markdown ---
path_to_file = "enwik8" #@param ["enwik4", "enwik6", "enwik8", "enwik9", "custom"]
#@markdown _The name of the file to compress or decompress. If "custom" is selected, use the next parameter to specify your custom file path._

#@markdown ---
custom_path = '' #@param {type:"string"}
#@markdown _Used when "path_to_file" is set to "custom". Specify the name of the file you want to compress or decompress. You can transfer files using the "http_path" or "local_upload" options below._

#@markdown ---
http_path = '' #@param {type:"string"}
#@markdown _Selects the URL to download your file from. Using Google Drive URLs is recommended for faster transfer speeds. Use the format "https://drive.google.com/uc?id=<FILE_ID>". You can find the file ID from the "Get Link" URL in Google Drive. You can enter multiple URLs separated by spaces._

#@markdown ---
local_upload = False #@param {type:"boolean"}
#@markdown _If enabled, prompts you in the "Setup Files" section to upload files directly from your computer. Note: Uploading locally can be slow compared to using "http_path"._

#@markdown ---
download_option = "no_download" #@param ["no_download", "local", "google_drive"]
#@markdown _Select how to handle output files. "local" downloads directly to your device. "google_drive" copies them to your Google Drive for much faster transfer speeds._

#@markdown ---
checkpoint = False #@param {type:"boolean"}
#@markdown _If enabled, a checkpoint of model weights will be saved according to your "download_option". This is useful for avoiding Colab session timeouts by splitting files into segments and saving/loading model weights between runs. Existing checkpoints load automatically at the start of compression._

#@markdown ---
total_parts = 1 #@param {type:"integer"}
#@markdown _The total number of segments to split file processing into across several sessions._

#@markdown ---
current_part = 1 #@param {type:"integer"}
#@markdown _The current file part to process (range: 1 to total_parts)._

#@markdown ---
retrain_period_schedule = "0:1001 200000:5001" #@param {type:"string"}
#@markdown _The retrain period schedule formatted as "step:value". Values are linearly interpolated between points._

#@markdown ---
retrain_block_len = 100000 #@param {type:"integer"}
#@markdown _Retrain over the last M symbols._

#@markdown ---
retrain_seq_length = 100 #@param {type:"integer"}
#@markdown _The sequence length used during retraining. This permits a longer horizon during retraining steps compared to online compression._

#@markdown ---
retrain_batch_size = 256 #@param {type:"integer"}
#@markdown _The batch size designated for retraining. Increasing this improves parallelism during the retraining phase._

#@markdown ---
retrain_lr_schedule = "0:0.0005 200000:0.0002" #@param {type:"string"}
#@markdown _The retraining learning rate schedule formatted as "step:value". Values are linearly interpolated between points._

#@markdown ---
retrain_dropout = 0.4 #@param {type:"number"}
#@markdown _The dropout rate applied during retraining._

## Setup

In [ ]:
#@title Imports

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import random
from google.colab import files
import time
import math
import sys
import subprocess
import contextlib
import os
from typing import Any
from google.colab import drive
import pickle
from flax.training import checkpoints as flax_checkpoints

# Set JAX to use 64-bit precision if needed, but for compression mixed/32 is likely intended.
# JAX defaults to 32-bit floats.
# os.environ['JAX_ENABLE_X64'] = '1'

In [ ]:
#@title System Info

def system_info():
  """Prints out system information."""
  gpu_info = !nvidia-smi
  gpu_info = '\n'.join(gpu_info)
  if gpu_info.find('failed') >= 0:
    print('Select the Runtime → "Change runtime type" menu to enable a GPU accelerator, ')
    print('and then re-execute this cell.')
  else:
    print(gpu_info)
  print("JAX version: ", jax.__version__)
  !lscpu |grep 'Model name'
  !cat /proc/meminfo | head -n 3

system_info()

In [ ]:
#@title Mount Google Drive
if download_option == "google_drive":
  drive.mount('/content/gdrive')

In [ ]:
#@title Setup Files

!mkdir -p "data"

if local_upload:
  %cd data
  files.upload()
  %cd ..

if path_to_file == 'enwik8' or path_to_file == 'enwik6' or path_to_file == 'enwik4':
  %cd data
  # Download enwik8 dataset
  !gdown --id 11-twesB-vGexGZpVSFbaCZDXKh5jMmMd
  # Create smaller subsets of the dataset
  !head -c 1000000 enwik8 > enwik6
  !head -c 10000 enwik8 > enwik4
  path_to_file = 'data/' + path_to_file
  %cd ..

if path_to_file == 'enwik9':
  %cd data
  # Download and extract the full enwik9 dataset
  !gdown --id 1D2gCmf9AlXIBP62ARhy0XcIuIolOTRAE
  !unzip enwik9.zip
  path_to_file = 'data/' + path_to_file
  %cd ..

if path_to_file == 'custom':
  path_to_file = 'data/' + custom_path

if http_path:
  %cd data
  paths = http_path.split()
  # Download any custom HTTP paths provided
  for path in paths:
    !gdown $path
  %cd ..

if preprocess == 'cmix':
  # Download and extract the cmix preprocessor
  !gdown --id 1qa7K28tlUDs9GGYbaL_iE9M4m0L1bYm9
  !unzip cmix-v18.zip
  %cd cmix
  !make
  %cd ..

if preprocess == 'nncp' or preprocess == 'nncp-done':
  # Download and extract the NNCP preprocessor
  !gdown --id 1VzFEzfN3Id0F32JygxelMPyOxGpIabL4
  !unzip nncp.zip
  %cd nncp/
  !make preprocess
  %cd ..

if checkpoint:
    ckpt_path = "data/ckpt.zip"
    if os.path.exists(ckpt_path):
        print(f"Unzipping model checkpoint from {ckpt_path}...")
        !unzip -o {ckpt_path} -d data/ckpt
    else:
        print(f"Checkpoint flag is enabled but {ckpt_path} was not found. Please upload it if you are resuming a session.")

if current_part > 1:
    partial_compressed_path = "data/compressed.dat"
    if not os.path.exists(partial_compressed_path):
        print(f"Warning: Attempting to resume compression (part {current_part}), but {partial_compressed_path} could not be found.")
        print("Please ensure you have uploaded the partially compressed file from your previous session.")
    else:
        print(f"Successfully found the partial compressed file: {partial_compressed_path}")

In [ ]:
#@title Model Architecture

class LSTMModel(nn.Module):
  vocab_size: int
  embedding_size: int
  rnn_units: int
  num_layers: int
  dropout_rate: float = 0.0
  dtype: Any = jnp.float32

  @nn.compact
  def __call__(self, inputs, states, deterministic=True, return_sequence=False):
    """
    Args:
      inputs: (batch, seq_length) int32
      states: list of tuples (c, h), length num_layers. Each c/h is size (batch, units).
              Note: Flax LSTMCell uses (carry, hidden) where the carry is (c, h).
      deterministic: boolean, if False, dropout is applied.
      return_sequence: boolean, if True, returns the output logits for the entire sequence.
    """
    # Embed the inputs
    x = nn.Embed(num_embeddings=self.vocab_size, features=self.embedding_size, dtype=self.dtype)(inputs)
    x = nn.Dropout(rate=self.dropout_rate, deterministic=deterministic)(x)
    # x shape: (batch, seq, embed)
    
    # Transpose for scanning over the time dimension: (seq, batch, embed)
    embedding_t = jnp.transpose(x, (1, 0, 2))
    
    new_states = []
    layer_outputs = [] # To store (seq, batch, units) outputs for each layer
    
    curr_input = embedding_t
    
    # Create the Scanned LSTM Cell Class
    # We scan over axis 0 of the input (time dimension)
    LSTMScan = nn.scan(nn.LSTMCell,
                       variable_broadcast="params",
                       split_rngs={"params": False},
                       in_axes=0, out_axes=0)

    for i in range(self.num_layers):
      # Instantiate the scanned LSTM layer
      lstm_scan = LSTMScan(self.rnn_units, dtype=self.dtype)
      
      carry = states[i] # (c, h)
      
      # Run the scan over the complete sequence
      new_carry, outputs = lstm_scan(carry, curr_input)
      
      # Apply dropout on the layer outputs, except for the last layer
      if i < self.num_layers - 1:
          outputs = nn.Dropout(rate=self.dropout_rate, deterministic=deterministic)(outputs)

      new_states.append(new_carry)
      layer_outputs.append(outputs)
      
      # Next layer input: Concatenate the embedding with the previous layer's output
      # Both are shape (seq, batch, ...)
      if i < self.num_layers - 1:
          curr_input = jnp.concatenate([embedding_t, outputs], axis=-1)
          
    # Sequence output logic
    if return_sequence:
        if self.num_layers == 1:
            final_rep = layer_outputs[0] # (seq, batch, units)
        else:
            final_rep = jnp.concatenate(layer_outputs, axis=-1) # (seq, batch, units * num_layers)
        
        # Pass representation through a dense layer
        logits = nn.Dense(self.vocab_size, name='dense_logits', dtype=self.dtype)(final_rep)
        
        # Transpose logits from shape (seq, batch, vocab) back to (batch, seq, vocab)
        logits = jnp.transpose(logits, (1, 0, 2))
    else:
        # Take the final timestep of each layer. 
        # layer_outputs[i] is shape (seq, batch, units), so [-1] yields (batch, units).
        last_timesteps = [out[-1] for out in layer_outputs] 
        
        if self.num_layers == 1:
          final_rep = last_timesteps[0]
        else:
          final_rep = jnp.concatenate(last_timesteps, axis=-1)
        
        # Pass final timestep representation through a dense layer
        logits = nn.Dense(self.vocab_size, name='dense_logits', dtype=self.dtype)(final_rep)
    
    # We return raw logits instead of probabilities for better numerical stability in the loss function.
    # Probabilities can be easily computed via jax.nn.softmax(logits).
    # Cast to float32 mainly for mixed precision stability
    logits = logits.astype(jnp.float32)
    
    return logits, new_states

In [ ]:
#@title Compression Library

from functools import partial
from flax.serialization import to_bytes, from_bytes

def parse_schedule(schedule_str):
  """Parses a learning rate or training schedule string.
  
  Expects a string formatted as "step:value step:value" and 
  returns a list of parsed (step, value) tuples sorted by step.
  """
  points = []
  try:
    for item in schedule_str.split():
      step, value = item.split(':')
      points.append((float(step), float(value)))
    points.sort(key=lambda x: x[0])
  except ValueError:
    print(f"Error parsing schedule: {schedule_str}")
    return []
  return points

def get_scheduled_value(schedule, step):
  """Linearly interpolates an exact value for a given step based on a predefined schedule.
  
  If the given step falls outside the scheduled bounds, the closest start or end 
  value is returned. Steps within scheduled intervals are interpolated linearly.
  """
  if not schedule:
    return 0.0
  
  if step <= schedule[0][0]:
    return schedule[0][1]
  
  if step >= schedule[-1][0]:
    return schedule[-1][1]
  
  # Linear search for the interval (optimized via bisect if needed, but linear is fine for small schedules)
  for i in range(len(schedule) - 1):
    start_step, start_val = schedule[i]
    end_step, end_val = schedule[i+1]
    
    if start_step <= step <= end_step:
      fraction = (step - start_step) / (end_step - start_step)
      return start_val + fraction * (end_val - start_val)
      
  return schedule[-1][1] # Should be unreachable

def get_symbol(index, length, freq, coder, compress, data):
  """Runs the arithmetic coder to encode or decode the next sequential symbol.

  During compression, retrieves the literal character from the known data and 
  encodes it into the bitstream using predicted probabilities. During decompression, 
  decodes the bitstream to determine the original character.

  Args:
    index: Int; the current positional index in the file.
    length: Int; the integer size limit (EOF) of the target file.
    freq: ndarray; predicted probabilities for all possible symbols.
    coder: ArithmeticCoder; the context coder driving the byte output.
    compress: Boolean; True if actively compressing, False for decompressing.
    data: List; contains all symbols in the file for read/write access.

  Returns:
    The next character symbol parsed, or 0 if `index` exceeds the file size limits.
  """
  symbol = 0
  if index < length:
    if compress:
      symbol = data[index]
      coder.write(freq, symbol)
    else:
      symbol = coder.read(freq)
      data[index] = symbol
  return symbol

def reset_seed():
  """Initializes various random seeds to help with determinism."""
  SEED = 1234
  os.environ['PYTHONHASHSEED']=str(SEED)
  random.seed(SEED)
  np.random.seed(SEED)

def download(path):
  """Downloads the file at the specified path."""
  if download_option == 'local':
    files.download(path)
  elif download_option == 'google_drive':
    !cp -f $path /content/gdrive/My\ Drive

@partial(jax.jit, static_argnames=['model'])
def predict_step(model, params, inputs, states):
    """Runs forward pass to get probabilities with ensemble geometric mean."""
    def forward_single(p, s):
        # Deterministic=True for prediction (no dropout)
        logits, new_s = model.apply(p, inputs, s, deterministic=True, return_sequence=False)
        return jax.nn.log_softmax(logits), new_s
    
    # Vectorize processing across the ensemble dimension (axis 0 of params and states).
    # Returns generated layer probabilities and updated neural states for each ensemble element.
    log_probs_ensemble, new_states = jax.vmap(forward_single, in_axes=(0, 0))(params, states)
    
    # Calculate geometric mean probabilities (normalized via softmax of the mean log_probs).
    return jax.nn.softmax(jnp.mean(log_probs_ensemble, axis=0)), new_states

@partial(jax.jit, static_argnames=['model', 'optimizer'])
def update_step(model, optimizer, params, opt_state, inputs, states, symbols, mask):
    """Runs forward+backward pass to update ensemble models and get next states."""
    
    def single_update(p, opt_s, s):
        def loss_func(params_single):
            # Disable dropout for online single-step updates.
            l_logits, l_states = model.apply(params_single, inputs, s, deterministic=True, return_sequence=False)
            one_hot = jax.nn.one_hot(symbols, model.vocab_size)
            
            # Compute cross-entropy loss.
            loss_vals = optax.softmax_cross_entropy(l_logits, one_hot)
            
            # Pack supplementary loss logic and metrics into the aux return.
            return jnp.sum(loss_vals * mask), (l_states, jax.nn.log_softmax(l_logits))
            
        (loss_val, (new_s, log_probs)), grads = jax.value_and_grad(loss_func, has_aux=True)(p)
        
        updates, new_opt_s = optimizer.update(grads, opt_s, p)
        new_p = optax.apply_updates(p, updates)
        return new_p, new_opt_s, new_s, log_probs

    # Vectorize the update across the entire ensemble.
    new_params, new_opt_state, new_states, all_log_probs = jax.vmap(
        single_update, in_axes=(0, 0, 0)
    )(params, opt_state, states)
    
    # Calculate aggregate ensemble loss based on geometric mean probabilities.
    mean_log_probs = jnp.mean(all_log_probs, axis=0) # (batch, vocab)
    
    one_hot = jax.nn.one_hot(symbols, model.vocab_size)
    # Cross entropy mapping from the normalized geometric values.
    loss_vals = optax.softmax_cross_entropy(mean_log_probs, one_hot)
    
    denom = jnp.sum(mask)
    return new_params, new_opt_state, new_states, jnp.sum(loss_vals * mask), denom

@partial(jax.jit, static_argnames=['model', 'optimizer'])
def retrain_step(model, optimizer, params, opt_state, inputs, targets, states, rngs, current_lr):
    """Executes a combined forward and backward pass over a sequence with dropout enabled.
    
    This function is used specifically during periodic retraining, calculating the 
    loss across the complete sequence instead of just a single timestep. Output 
    probabilities are regularized with dropout.
    """
    
    def single_retrain(p, opt_s, s, rng):
        def loss_func(params_single):
            # Compute loss across the entire sequence length with targeted dropout enablement.
            l_logits, l_states = model.apply(params_single, inputs, s, 
                                           deterministic=False, return_sequence=True, 
                                           rngs={'dropout': rng})
                                           
            one_hot = jax.nn.one_hot(targets, model.vocab_size)
            loss_vals = optax.softmax_cross_entropy(l_logits, one_hot)
            return jnp.mean(loss_vals), l_states
            
        (loss_val, new_s), grads = jax.value_and_grad(loss_func, has_aux=True)(p)
        
        updates, new_opt_s = optimizer.update(grads, opt_s, p)
        
        # Scale updates
        updates = jax.tree.map(lambda u: u * current_lr, updates)
        
        new_p = optax.apply_updates(p, updates)
        return new_p, new_opt_s, new_s, loss_val

    # Vectorize the retraining batch process iteratively across the ensemble dimension.
    new_params, new_opt_state, new_states, losses = jax.vmap(
        single_retrain, in_axes=(0, 0, 0, 0)
    )(params, opt_state, states, rngs)
    
    return new_params, new_opt_state, new_states, jnp.mean(losses)

def process(compress, length, vocab_size, coder, data):
  """Main processing loop for data compression and decompression.

  This function iteratively streams characters into the neural network, 
  updating internal weights, managing training schedules, and coordinating 
  the arithmetic coder.

  Args:
    compress: Boolean; True if compressing a file, False if decompressing.
    length: Int; total size limit of the file being processed.
    vocab_size: Int; numerical size of the expected character vocabulary.
    coder: ArithmeticCoder; handles the byte stream conversion mapping.
    data: List; contains characters/symbols inside the file payload.
  """
  start = time.time()
  last_print_time = start
  reset_seed()

  # Parse Schedules
  lr_schedule = parse_schedule(learning_rate_schedule)
  retrain_p_schedule = parse_schedule(retrain_period_schedule)
  retrain_l_schedule = parse_schedule(retrain_lr_schedule)
  
  # Parameter Summary
  print(f"batch_size={batch_size}, seq_length={seq_length}, rnn_units={rnn_units}, num_layers={num_layers}, "
        f"embedding_size={embedding_size}, ensemble_size={ensemble_size}, learning_rate_schedule={learning_rate_schedule}, "
        f"adam_b1={adam_b1}, adam_b2={adam_b2}, adam_eps={adam_eps}, "
        f"retrain_period_schedule={retrain_period_schedule}, "
        f"retrain_block_len={retrain_block_len}, retrain_seq_length={retrain_seq_length}, "
        f"retrain_batch_size={retrain_batch_size}, retrain_lr_schedule={retrain_lr_schedule}, "
        f"retrain_dropout={retrain_dropout}, total_parts={total_parts}, current_part={current_part}")
  
  # Initialize JAX RNG
  rng_key = jax.random.PRNGKey(1234)
  rng_key, init_key, retrain_key = jax.random.split(rng_key, 3)
  
  # Use bfloat16 for TPU efficiency
  model_dtype = jnp.bfloat16

  model = LSTMModel(vocab_size=vocab_size, embedding_size=embedding_size, 
                    rnn_units=rnn_units, num_layers=num_layers, 
                    dropout_rate=retrain_dropout, dtype=model_dtype)
  
  # Initialize Model Parameters
  dummy_input = jnp.zeros((batch_size, seq_length), dtype=jnp.int32)
  # Flax LSTMCell state is (carry, hidden).
  # If model is in bfloat16, carry/hidden should be castable or handled.
  # We initialize with float32 zeros, Flax/JAX handles dtype promotion/cast in layer.
  dummy_states = [(jnp.zeros((batch_size, rnn_units)), jnp.zeros((batch_size, rnn_units))) for _ in range(num_layers)]
  
  # Ensemble Initialization
  keys = jax.random.split(init_key, ensemble_size)
  
  def init_single(key):
     return model.init(key, dummy_input, dummy_states)
     
  params = jax.vmap(init_single)(keys)
  
  # Check for checkpoint to restore
  if checkpoint:
      ckpt_dir = os.path.abspath('data/ckpt')
      # Flax checkpoint restores the structure if target is provided.
      # If no checkpoint exists, it returns target (params).
      params = flax_checkpoints.restore_checkpoint(ckpt_dir=ckpt_dir, target=params)
      # Note: restore_checkpoint does not fail if no checkpoint is found (unless step is specified),
      # it simply returns the target structure.
      # We check if a checkpoint was actually restored by looking at folder content for logging
      if flax_checkpoints.latest_checkpoint(ckpt_dir):
          print("Restored model weights from checkpoint.")
      else:
          print("No checkpoint found. Starting from scratch.")

  # Model Summary
  print("\n" + "="*80)
  print(f"Model Architecture (Ensemble Size: {ensemble_size})")
  print("="*80)
  
  # Print summary of a single model using tabulate
  # Note: tabulate was added in Flax 0.4.1. If not available, we skip.
  try:
      # Use a dummy key for tabulation
      tabulate_fn = nn.tabulate(model, jax.random.PRNGKey(0), console_kwargs={'width': 100})
      print(tabulate_fn(dummy_input, [(jnp.zeros((batch_size, rnn_units)), jnp.zeros((batch_size, rnn_units))) for _ in range(num_layers)]))
  except Exception as e:
      print(f"Could not print detailed summary table: {e}")

  # Total Parameters for Ensemble
  total_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
  print(f"\nTotal Ensemble Parameters: {total_params:,}")
  print("="*80 + "\n")
  
  # Replicate dummy_states for ensemble
  dummy_states_single = dummy_states
  dummy_states = jax.tree.map(lambda x: jnp.broadcast_to(x, (ensemble_size,) + x.shape), dummy_states)
  
  # Optimizer
  split = math.ceil(length / batch_size)

  if lr_schedule:
      lr_steps = jnp.array([p[0] for p in lr_schedule], dtype=jnp.float32)
      lr_values = jnp.array([p[1] for p in lr_schedule], dtype=jnp.float32)
      scheduler = lambda step: jnp.interp(step, lr_steps, lr_values, left=lr_values[0], right=lr_values[-1])
  else:
      scheduler = lambda step: 0.0005 # Fallback
      
  optimizer = optax.chain(
      optax.clip_by_global_norm(4.0),
      optax.adam(learning_rate=scheduler, b1=adam_b1, b2=adam_b2, eps=adam_eps)
  )
  # Initialize optimizer state for ensemble
  opt_state = jax.vmap(optimizer.init)(params)

  # Retraining Optimizer
  # Defined globally with constant LR=1.0 to avoid recompilation
  retrain_optimizer = optax.chain(
      optax.clip_by_global_norm(4.0),
      optax.adam(learning_rate=1.0, b1=adam_b1, b2=adam_b2, eps=adam_eps)
  )
  retrain_opt_state = jax.vmap(retrain_optimizer.init)(params)
  
  # Define partitioned step limitations
  part_len = math.ceil(split / total_parts)
  start_pos = (current_part - 1) * part_len
  end_pos = min(start_pos + part_len, split)
  print(f"Processing part {current_part} of {total_parts}. Step range: {start_pos} to {end_pos - 1} (Total split: {split})")
  
  # Initialize static probabilities for the zero-state prior logic.
  freq = np.cumsum(np.full(vocab_size, (1.0 / vocab_size)) * 10000000 + 1)
  
  pos = 0
  
  # State Loading / Warmup Context Checking
  current_states = dummy_states
  
  model_state_loaded = False
  if start_pos > 0 and checkpoint:
      ckpt_dir = os.path.abspath('data/ckpt')
      model_state_path = os.path.join(ckpt_dir, 'model_state.pkl')
      
      if os.path.exists(model_state_path):
          print("Attempting to load model states from checkpoint...")
          try:
              with open(model_state_path, 'rb') as f:
                  loaded_state = pickle.load(f)
              
              # Reconstruct cached model histories scaling the internal loop sequence cell logic.
              states_queue_loaded = loaded_state['states_queue']
              seq_input_loaded = loaded_state['seq_input']
              
              # Enforce sequence structure tracking validations upon target.
              if seq_input_loaded.shape != (batch_size, seq_length):
                  print(f"Warning: Loaded seq_input shape {seq_input_loaded.shape} mismatch with batch/seq params.")
                  raise ValueError("Shape mismatch")
                  
              seq_input = jnp.array(seq_input_loaded)
              
              # Recursively map nested target dictionary variables natively into supported array shapes.
              states_queue = jax.tree.map(jnp.array, states_queue_loaded)
              
              # Safely re-initialize Optax model states preserving variable precision tracking limits
              if 'opt_state_bytes' in loaded_state:
                  opt_state = from_bytes(opt_state, loaded_state['opt_state_bytes'])
                  print("Restored main optimizer state.")
              
              if 'retrain_opt_state_bytes' in loaded_state:
                  retrain_opt_state = from_bytes(retrain_opt_state, loaded_state['retrain_opt_state_bytes'])
                  print("Restored retraining optimizer state.")

              print(f"Successfully loaded model states. Skipping warmup.")
              model_state_loaded = True
              pos = start_pos
              
          except Exception as e:
              print(f"Failed to load model states: {e}. Falling back to warmup.")
  
  if start_pos > 0 and not model_state_loaded:
      warmup_len = 500
      run_start_pos = max(0, start_pos - warmup_len)
      
      # Re-build `seq_input` for `run_start_pos`.
      w_symbols = []
      for i in range(batch_size):
           # current pos `run_start_pos`.
           start_idx_window = run_start_pos - seq_length + 1
           batch_syms = []
           for k in range(seq_length):
               idx_k = start_idx_window + k # Goes from `start` to `start + L - 1 = run_start_pos`
               s_idx = idx_k + i*split
               # Read data
               if s_idx < 0:
                   # For i=0 stream, early index
                   val = data[i*split] if i*split < len(data) else 0 # Fallback
               else:
                   val = data[s_idx] if s_idx < len(data) else 0
               batch_syms.append(val)
           w_symbols.append(batch_syms)
      seq_input = jnp.array(w_symbols, dtype=jnp.int32)
      
      states_queue = [dummy_states] * seq_length # Reset
      
      # Run the warmup
      print(f"Warming up states from {run_start_pos} to {start_pos}...")
      for w_pos in range(run_start_pos, start_pos):
           state_in = states_queue.pop(0)
           _, new_states = predict_step(model, params, seq_input, state_in)
           states_queue.append(new_states)
           
           # Update input window
           next_in_symbols = []
           for i in range(batch_size):
               # Next symbol index: w_pos + 1 + i*split
               idx = w_pos + 1 + i * split
               val = data[idx] if idx < len(data) else 0
               next_in_symbols.append(val)
           next_in_jax = jnp.expand_dims(jnp.array(next_in_symbols, dtype=jnp.int32), 1)
           seq_input = jnp.concatenate([seq_input[:, 1:], next_in_jax], axis=1)

      pos = start_pos
      
  elif not model_state_loaded:
      # Default initialization
      initial_symbols = []
      for i in range(batch_size):
        initial_symbols.append(get_symbol(i*split, length, freq, coder, compress, data))
      seq_input = jnp.tile(jnp.expand_dims(jnp.array(initial_symbols, dtype=jnp.int32), 1), (1, seq_length))
      states_queue = [dummy_states] * seq_length
      pos = 0

  cross_entropy = 0.0 
  denom = 0.0
  last_output = 0
  template = '{:0.2f}%\tcross entropy: {:0.2f}\ttime: {:0.2f}\tlr: {:0.8f}\tstep: {}'
  
  # Pre-compile functions with static args
  predict_fn = partial(predict_step, model)
  update_fn = partial(update_step, model, optimizer)
  retrain_fn = partial(retrain_step, model, retrain_optimizer)

  last_retrain_pos = 0

  while pos < end_pos:

    # Calculate current retrain params
    current_retrain_period = get_scheduled_value(retrain_p_schedule, pos)
    current_retrain_lr = get_scheduled_value(retrain_l_schedule, pos)

    # Check for periodic training iterations over batch contexts
    if current_retrain_period > 0 and (pos - last_retrain_pos) >= current_retrain_period:
        retrain_start_time = time.time()
        
        # Determine internal context parameters corresponding to backward schedule updates
        r_start_step = max(0, pos - retrain_block_len)
        r_end_step = pos
        
        # Buffer arrays resolving sequential dependencies required for super-batch training inputs.
        all_inputs = []
        all_targets = []
        
        r_step = r_start_step
        while r_step < r_end_step:
            for i in range(batch_size):
                base_idx = r_step + i * split 
                # Determine absolute data bounds relative to the current stream chunk.
                # Valid segments span linearly to prevent index bounds overflow during training loops.
                
                start_idx = base_idx
                end_idx = start_idx + retrain_seq_length + 1
                current_stream_limit = i * split + pos + 1

                stream_segment = data[start_idx : min(end_idx, current_stream_limit)]
                
                # Pad sequences if a chunk bounds abruptly to ensure stable batch dimensional scaling.
                if len(stream_segment) < retrain_seq_length + 1:
                    stream_segment += [0] * (retrain_seq_length + 1 - len(stream_segment))
                
                all_inputs.append(stream_segment[:-1])
                all_targets.append(stream_segment[1:])
            
            r_step += retrain_seq_length
            
        # Convert list segments into strongly typed array structures suitable for JAX processing calculations.
        all_inputs_jax = jnp.array(all_inputs, dtype=jnp.int32)
        all_targets_jax = jnp.array(all_targets, dtype=jnp.int32)
        
        total_examples = all_inputs_jax.shape[0]
        
        # Drop any hanging segment reminders gracefully to prevent redundant and costly JIT compilations
        remainder = total_examples % retrain_batch_size
        if remainder != 0:
            # Drop the last 'remainder' examples
            all_inputs_jax = all_inputs_jax[:-remainder]
            all_targets_jax = all_targets_jax[:-remainder]
            total_examples -= remainder

        print(f"Starting retraining at step {pos}... (period={current_retrain_period:.1f}, lr={current_retrain_lr:.8f}, examples={total_examples})")

        if total_examples > 0:
            # Cast current_retrain_lr to a JAX array to prevent recompilation from Python float changes
            curr_lr_jax = jnp.array(current_retrain_lr, dtype=jnp.float32)

            # Iterate over super-batches
            for i in range(0, total_examples, retrain_batch_size):
                batch_inputs = all_inputs_jax[i : i + retrain_batch_size]
                batch_targets = all_targets_jax[i : i + retrain_batch_size]
                
                # Initial states for this batch (zeros)
                # Shape for one layer state component: (ensemble_size, retrain_batch_size, rnn_units)
                batch_states = [
                    (jnp.zeros((ensemble_size, retrain_batch_size, rnn_units)), 
                     jnp.zeros((ensemble_size, retrain_batch_size, rnn_units))) 
                    for _ in range(num_layers)
                ]
                                
                # Generate dropout keys
                retrain_key, dropout_key = jax.random.split(retrain_key)
                dropout_keys = jax.random.split(dropout_key, ensemble_size)

                params, retrain_opt_state, _, r_loss = retrain_fn(
                    params, retrain_opt_state, batch_inputs, batch_targets, batch_states, dropout_keys, curr_lr_jax
                )

        last_retrain_pos = pos
        print(f"Retraining finished. Duration: {time.time() - retrain_start_time:.2f}s")

    state_in = states_queue.pop(0)

    # 1. Predict
    probs, _ = predict_fn(params, seq_input, state_in)
    
    # 2. Arithmetic Coding (CPU)
    # Convert optimized JAX array to numpy for CPU processing
    p = np.array(probs)
    
    current_symbols = []
    current_mask = []
    
    for i in range(batch_size):
        p_i = p[i]
        freq = np.cumsum(p_i * 10000000 + 1)
        index = pos + 1 + i * split
        
        symbol = get_symbol(index, length, freq, coder, compress, data)
        current_symbols.append(symbol)
        
        if index < length:
            current_mask.append(1.0)
        else:
            current_mask.append(0.0)
            
    symbols_jax = jnp.array(current_symbols, dtype=jnp.int32)
    
    # 3. Update Model
    mask_jax = jnp.array(current_mask, dtype=jnp.float32)

    params, opt_state, new_states, loss_val, loss_denom = update_fn(
        params, opt_state, seq_input, state_in, symbols_jax, mask_jax
    )
    
    # 4. Update State Queue
    states_queue.append(new_states)
    
    # 5. Update Metrics
    cross_entropy += loss_val
    denom += loss_denom
    
    # 6. Prepare for next step
    # Shift seq_input: discard first col, append new symbol
    # seq_input shape: (batch, seq_length)
    # symbols_jax shape: (batch,) -> expand to (batch, 1)
    
    seq_input = jnp.concatenate([seq_input[:, 1:], jnp.expand_dims(symbols_jax, 1)], axis=1)

    pos += 1
    
    if time.time() - last_print_time >= 20:
       last_print_time = time.time()
       time_diff = last_print_time - start
       current_bpc = (cross_entropy / denom) / np.log(2)
       current_lr = scheduler(pos)
       print(template.format(pos / split * 100, current_bpc, time_diff, current_lr, pos))
  
  if checkpoint:
      # Pre-emptively allocate explicit directory buffers to prevent target failure upon session interruptions.
      print("Saving checkpoint...")
      
      ckpt_dir = os.path.abspath('data/ckpt')
      if not os.path.exists(ckpt_dir):
        os.makedirs(ckpt_dir)
      
      flax_checkpoints.save_checkpoint(ckpt_dir=ckpt_dir, target=params, step=0, overwrite=True)
      print("Checkpoint saved.")
      
      # Serialize contextual variables resolving internal coder pointer alignments dynamically.
      ac_state = {
          'coder': coder.get_state(),
          'bitstream': coder.output.get_state() if compress else coder.input.get_state()
      }
      
      # Implicit read position streams must formally track file object indices sequentially across chunks.
      if not compress:
           ac_state['file_pos'] = coder.input.input.tell()
           
      with open(os.path.join(ckpt_dir, 'ac_state.pkl'), 'wb') as f:
          pickle.dump(ac_state, f)
      print("AC state saved.")
      
      # Explicit cache serialization for active iteration inputs and sequence cell tracking components.
      
      # Type downcast to traditional numpy mappings to sidestep complex pickling abstraction failures.
      seq_input_np = np.array(seq_input)
      states_queue_np = jax.tree.map(np.array, states_queue)
      
      # Delegate nested Optax learning configurations internally resolving native memory parameters robustly safely.
      
      opt_state_bytes = to_bytes(opt_state)
      retrain_opt_state_bytes = to_bytes(retrain_opt_state)
      
      model_state = {
          'seq_input': seq_input_np,
          'states_queue': states_queue_np,
          'opt_state_bytes': opt_state_bytes,
          'retrain_opt_state_bytes': retrain_opt_state_bytes
      }
      with open(os.path.join(ckpt_dir, 'model_state.pkl'), 'wb') as f:
          pickle.dump(model_state, f)
      print("Model state (including optimizers) saved.")

In [ ]:
#@title Arithmetic Coding Library

#
# Reference arithmetic coding
# Copyright (c) Project Nayuki
#
# https://www.nayuki.io/page/reference-arithmetic-coding
# https://github.com/nayuki/Reference-arithmetic-coding
#

import sys
python3 = sys.version_info.major >= 3


# ---- Arithmetic coding core classes ----

# Provides the state and behaviors that arithmetic coding encoders and decoders share.
class ArithmeticCoderBase(object):

	# Constructs an arithmetic coder, which initializes the internal code range.
	def __init__(self, numbits):
		if numbits < 1:
			raise ValueError("State size out of range")

		# -- Configuration fields --
		# Number of bits for the 'low' and 'high' state variables. Must be at least 1.
		self.num_state_bits = numbits
		# Maximum range (high+1-low) during coding (trivial), which is 2^num_state_bits = 1000...000.
		self.full_range = 1 << self.num_state_bits
		# The top bit at width num_state_bits, which is 0100...000.
		self.half_range = self.full_range >> 1
		# The second highest bit at width num_state_bits, which is 0010...000. This is zero when num_state_bits=1.
		self.quarter_range = self.half_range >> 1
		# Minimum range (high+1-low) during coding (non-trivial), which is 0010...010.
		self.minimum_range = self.quarter_range + 2
		# Maximum allowed total from a frequency table at all times during coding.
		self.maximum_total = self.minimum_range
		# Bit mask of num_state_bits ones, which is 0111...111.
		self.state_mask = self.full_range - 1

		# -- State fields --
		# Low end of this arithmetic coder's current range.
		self.low = 0
		# High end of this arithmetic coder's current range.
		self.high = self.state_mask
        
	def get_state(self):
		return {
			'low': self.low,
			'high': self.high
		}
	
	def set_state(self, state):
		self.low = state['low']
		self.high = state['high']


	# Updates the code range (low and high) of this arithmetic coder as a result
	# of processing the given symbol with the given frequency table.
	def update(self, freqs, symbol):
		# State check
		low = self.low
		high = self.high
		range = high - low + 1

		# Frequency table values check
		total = int(freqs[-1])
		symlow = int(freqs[symbol-1]) if symbol > 0 else 0
		symhigh = int(freqs[symbol])

		# Update range
		newlow  = low + symlow  * range // total
		newhigh = low + symhigh * range // total - 1
		self.low = newlow
		self.high = newhigh

		# While low and high have the same top bit value, shift them out
		while ((self.low ^ self.high) & self.half_range) == 0:
			self.shift()
			self.low  = ((self.low  << 1) & self.state_mask)
			self.high = ((self.high << 1) & self.state_mask) | 1
		# Now low's top bit must be 0 and high's top bit must be 1

		# While low's top two bits are 01 and high's are 10, delete the second highest bit of both
		while (self.low & ~self.high & self.quarter_range) != 0:
			self.underflow()
			self.low = (self.low << 1) ^ self.half_range
			self.high = ((self.high ^ self.half_range) << 1) | self.half_range | 1


	# Called to handle the situation when the top bit of 'low' and 'high' are equal.
	def shift(self):
		raise NotImplementedError()


	# Called to handle the situation when low=01(...) and high=10(...).
	def underflow(self):
		raise NotImplementedError()


class ArithmeticEncoder(ArithmeticCoderBase):

	# Constructs an arithmetic coding encoder based on the given bit-level output stream.
	def __init__(self, numbits, bitout):
		super(ArithmeticEncoder, self).__init__(numbits)
		# The underlying bit output stream.
		self.output = bitout
		# Number of saved underflow bits. This value can grow without bound.
		self.num_underflow = 0
		
	def get_state(self):
		state = super(ArithmeticEncoder, self).get_state()
		state['num_underflow'] = self.num_underflow
		return state
	
	def set_state(self, state):
		super(ArithmeticEncoder, self).set_state(state)
		self.num_underflow = state['num_underflow']


	# Encodes the given symbol based on the given frequency table.
	# This updates this arithmetic coder's state and may write out some bits.
	def write(self, freqs, symbol):
		self.update(freqs, symbol)


	# Terminates the arithmetic coding by flushing any buffered bits, so that the output can be decoded properly.
	# It is important that this method must be called at the end of the each encoding process.
	def finish(self):
		self.output.write(1)


	def shift(self):
		bit = self.low >> (self.num_state_bits - 1)
		self.output.write(bit)

		# Write out the saved underflow bits
		for _ in range(self.num_underflow):
			self.output.write(bit ^ 1)
		self.num_underflow = 0


	def underflow(self):
		self.num_underflow += 1


# Reads from an arithmetic-coded bit stream and decodes the stream back into symbols.
class ArithmeticDecoder(ArithmeticCoderBase):

	# Constructs an arithmetic coding decoder based on the
	# given bit input stream, and fills the initial code bits.
	def __init__(self, numbits, bitin):
		super(ArithmeticDecoder, self).__init__(numbits)
		# The underlying bit input stream.
		self.input = bitin
		# The current raw code bits being buffered, which is always in the range [low, high].
		self.code = 0
		for _ in range(self.num_state_bits):
			self.code = self.code << 1 | self.read_code_bit()
	
	def get_state(self):
		state = super(ArithmeticDecoder, self).get_state()
		state['code'] = self.code
		return state
	
	def set_state(self, state):
		super(ArithmeticDecoder, self).set_state(state)
		self.code = state['code']


	# Decodes the next symbol based on the given frequency table and returns it.
	# Also updates this arithmetic coder's state and may read in some bits.
	def read(self, freqs):

		# Translate from coding range scale to frequency table scale
		total = int(freqs[-1])
		range = self.high - self.low + 1
		offset = self.code - self.low
		value = ((offset + 1) * total - 1) // range

		# A kind of binary search. Find highest symbol such that freqs[symbol-1] <= value.
		start = 0
		end = len(freqs)
		while end - start > 1:
			middle = (start + end) >> 1
			low = int(freqs[middle-1]) if middle > 0 else 0
			if low > value:
				end = middle
			else:
				start = middle

		symbol = start
		self.update(freqs, symbol)
		return symbol


	def shift(self):
		self.code = ((self.code << 1) & self.state_mask) | self.read_code_bit()


	def underflow(self):
		self.code = (self.code & self.half_range) | ((self.code << 1) & (self.state_mask >> 1)) | self.read_code_bit()


	# Returns the next bit (0 or 1) from the input stream.
	def read_code_bit(self):
		temp = self.input.read()
		if temp == -1:
			temp = 0
		return temp


# ---- Bit-oriented I/O streams ----

# A stream of bits that can be read. The bits are read in big-endian order.
class BitInputStream(object):

	# Constructs a bit input stream based on the given byte input stream.
	def __init__(self, inp):
		# The underlying byte stream to read from
		self.input = inp
		# Either in the range [0x00, 0xFF] if bits are available, or -1 if end of stream is reached
		self.currentbyte = 0
		# Number of remaining bits in the current byte, always between 0 and 7 (inclusive)
		self.numbitsremaining = 0
  
	def get_state(self):
		return {
			'currentbyte': self.currentbyte,
			'numbitsremaining': self.numbitsremaining
		}
	
	def set_state(self, state):
		self.currentbyte = state['currentbyte']
		self.numbitsremaining = state['numbitsremaining']


	# Reads a bit from this stream. Returns 0 or 1 if a bit is available, or -1 if EOF.
	def read(self):
		if self.currentbyte == -1:
			return -1
		if self.numbitsremaining == 0:
			temp = self.input.read(1)
			if len(temp) == 0:
				self.currentbyte = -1
				return -1
			self.currentbyte = temp[0] if python3 else ord(temp)
			self.numbitsremaining = 8
		assert self.numbitsremaining > 0
		self.numbitsremaining -= 1
		return (self.currentbyte >> self.numbitsremaining) & 1


	# Reads a bit from this stream. Returns 0 or 1, or raises an EOFError if EOF.
	def read_no_eof(self):
		result = self.read()
		if result != -1:
			return result
		else:
			raise EOFError()


	# Closes this stream and the underlying input stream.
	def close(self):
		self.input.close()
		self.currentbyte = -1
		self.numbitsremaining = 0


# A stream where bits can be written. Padded with 0's to align on byte boundaries.
class BitOutputStream(object):

	# Constructs a bit output stream based on the given byte output stream.
	def __init__(self, out):
		self.output = out  # The underlying byte stream to write to
		self.currentbyte = 0  # The accumulated bits for the current byte
		self.numbitsfilled = 0  # Number of accumulated bits in the current byte

	def get_state(self):
		return {
			'currentbyte': self.currentbyte,
			'numbitsfilled': self.numbitsfilled
		}
	
	def set_state(self, state):
		self.currentbyte = state['currentbyte']
		self.numbitsfilled = state['numbitsfilled']

	# Writes a single bit to the stream.
	def write(self, b):
		if b not in (0, 1):
			raise ValueError("Argument must be 0 or 1")
		self.currentbyte = (self.currentbyte << 1) | b
		self.numbitsfilled += 1
		if self.numbitsfilled == 8:
			towrite = bytes((self.currentbyte,)) if python3 else chr(self.currentbyte)
			self.output.write(towrite)
			self.currentbyte = 0
			self.numbitsfilled = 0


	# Closes this stream. Writes padding to complete the last byte if partially filled.
	def close(self):
		while self.numbitsfilled != 0:
			self.write(0)
		self.output.close()

## Compress

In [ ]:
#@title Preprocess

if mode != 'decompress':
  input_path = path_to_file

  if preprocess == 'cmix':
    !./cmix/cmix -s ./cmix/dictionary/english.dic $path_to_file ./data/preprocessed.dat
    input_path = "./data/preprocessed.dat"

  # Parse the file characters into the sequential int_list buffer.
  int_list = []
  if preprocess == 'nncp' or preprocess == 'nncp-done':
    if preprocess == 'nncp':
      !time ./nncp/preprocess c data/dictionary.words $path_to_file data/preprocessed.dat $n_words $min_freq
    else:
      !cp $path_to_file data/preprocessed.dat
    input_path = "./data/preprocessed.dat"
    orig = open(input_path, 'rb').read()
    for i in range(0, len(orig), 2):
      int_list.append(orig[i] * 256 + orig[i+1])
    vocab_size = int(subprocess.check_output(
        ['wc', '-l', 'data/dictionary.words']).split()[0])
  else:
    text = open(input_path, 'rb').read()
    vocab = sorted(set(text))
    vocab_size = len(vocab)
    # Create a mapping from unique characters to sequential indexes.
    char2idx = {u:i for i, u in enumerate(vocab)}
    for idx, c in enumerate(text):
      int_list.append(char2idx[c])

  # Round the vocabulary size up to a multiple of 8 to improve processing performance.
  vocab_size = math.ceil(vocab_size/8) * 8
  
  if checkpoint:
      ckpt_dir = 'data/ckpt'
      if not os.path.exists(ckpt_dir):
          os.makedirs(ckpt_dir)
  else:
      ckpt_dir = None

  file_len = len(int_list)
  print('Length of file: {} symbols'.format(file_len))
  print('Vocabulary size: {}'.format(vocab_size))

In [ ]:
#@title Compression

if mode == 'compress' or mode == 'both':
  original_file = path_to_file
  path_to_file = "data/compressed.dat"
  
  # Determine correct file operational mode depending on checkpoint part status
  if current_part > 1:
      open_mode = "ab"
      print(f"Resuming compression (part {current_part}/{total_parts}). Appending to {path_to_file}...")
  else:
      open_mode = "wb"
      print(f"Starting compression (part {current_part}/{total_parts})...")

  with open(path_to_file, open_mode) as out:
      # Initialize BitOutputStream. State is tracked via bitout variables so we won't overwrite bits on resume.
      bitout = BitOutputStream(out)
      
      enc = ArithmeticEncoder(32, bitout)

      # Restore previous ArithmeticCoder state if resuming from part > 1.
      if current_part > 1:
          ckpt_dir = os.path.abspath('data/ckpt')
          ac_state_path = os.path.join(ckpt_dir, 'ac_state.pkl')
          if os.path.exists(ac_state_path):
              with open(ac_state_path, 'rb') as f:
                  ac_state = pickle.load(f)
              enc.set_state(ac_state['coder'])
              bitout.set_state(ac_state['bitstream'])
              print("Restored AC state.")
          else:
              print("Warning: Context AC state not found for resuming! Expected corruption.")

      length = len(int_list)
      
      # Prepare and write the overarching file header data if starting fresh
      if current_part == 1:
          # Write the original decoded file length to the compressed file header.
          out.write(length.to_bytes(5, byteorder='big', signed=False))
          if preprocess != 'nncp' and preprocess != 'nncp-done':
              # Without NNCP, explicitly allocate 256 bits for dynamic vocabulary tracking.
              for i in range(256):
                  if i in char2idx:
                      bitout.write(1)
                  else:
                      bitout.write(0)

      # Trigger main compression sequence
      process(True, length, vocab_size, enc, int_list)
      
      # Finish up the bitstream encoding only if this is the final processing chunk.
      if current_part == total_parts:
          enc.finish()
          bitout.close() # Flush and clear padding
      else:
          # On pause, hold off safely closing the bitout to avoid injecting padding bits.
          print(f"Pausing compression part {current_part}. AC state safely packaged for next execution.")
          pass

  print("Compressed size:", os.path.getsize(path_to_file))

In [ ]:
#@title Download Result

if mode == 'preprocess_only':
  if preprocess == 'nncp':
    download('data/dictionary.words')
  download(input_path)
elif mode != 'decompress':
  download('data/compressed.dat')
  if preprocess == 'nncp':
    download('data/dictionary.words')
  if checkpoint and mode != "both":
    # flax.training.checkpoints creates files like "checkpoint_0"
    # We should zip or download the directory
    import shutil
    shutil.make_archive('data/ckpt', 'zip', 'data/ckpt')
    download('data/ckpt.zip')

## Decompress

In [ ]:
#@title Decompression

if mode == 'decompress' or mode == 'both':
  output_path = "data/decompressed.dat"
  
  # Check for appropriate resuming status and partial files
  if current_part > 1:
      if not os.path.exists(output_path):
          print(f"Error: Attempting to resume decompression (part {current_part}), but {output_path} could not be found.")
          print("Please ensure the partial decompressed file was uploaded successfully.")
      else:
           print(f"Resuming decompression (part {current_part}/{total_parts}). Targetting {output_path}...")
  else:
      print(f"Starting decompression (part {current_part}/{total_parts})...")

  # Open the binary input path to interpret compressed sequence arrays
  with open(path_to_file, "rb") as inp:
      # Read the expected original file chunk length from the 5-byte header limit.
      length = int.from_bytes(inp.read()[:5], byteorder='big')
      
      # Allocate and prep the resulting symbol character buffer.
      output = [0] * length
      if current_part > 1:
           
           print("Loading partially decompressed context block...")
           if os.path.exists(output_path):
                # Retrieve standard file contents for context continuation matching.
                with open(output_path, "rb") as f_partial:
                    partial_data = f_partial.read()
                    
                # Reconstruct values safely mapping directly onto pre-processor logic.
                if preprocess == 'nncp' or preprocess == 'nncp-done':
                    for i in range(len(partial_data) // 2):
                        val = partial_data[2*i] * 256 + partial_data[2*i+1]
                        output[i] = val
                else:
                    # Decompressor will map reverse indices further downstream
                    pass 

      inp.seek(5)
      bitin = BitInputStream(inp)

      if preprocess == 'nncp' or preprocess == 'nncp-done':
        # Rely on fixed, precompiled dictionary vocab sizes via NNCP
        vocab_size = int(subprocess.check_output(
            ['wc', '-l', 'data/dictionary.words']).split()[0])
      else:
        # Check standard 256 byte matrix blocks to calculate explicit length
        vocab = []
        for i in range(256):
          if bitin.read():
            vocab.append(i)
        vocab_size = len(vocab)
        
        # Determine strict char2idx conversions mapping directly on file updates
        if current_part > 1 and not (preprocess == 'nncp' or preprocess == 'nncp-done'):
             char2val = {c: i for i, c in enumerate(vocab)}
             
             # Safely reverse match the expected character outputs
             for i in range(len(partial_data)):
                 c = partial_data[i]
                 if c in char2val:
                     output[i] = char2val[c]
                 else:
                     pass


      # Round up vocabulary limits tracking to multiples of 8 for pipeline efficiency.
      vocab_size = math.ceil(vocab_size/8) * 8
      
      dec = ArithmeticDecoder(32, bitin)
      
      # Refresh AC configurations from preserved memory checkpoints.
      if current_part > 1:
          ckpt_dir = os.path.abspath('data/ckpt')
          ac_state_path = os.path.join(ckpt_dir, 'ac_state.pkl')
          
          if os.path.exists(ac_state_path):
              with open(ac_state_path, 'rb') as f:
                  ac_state = pickle.load(f)
              
              dec.set_state(ac_state['coder'])
              bitin.set_state(ac_state['bitstream'])
              
              # Re-align raw file buffer offsets specifically targeted by stream positions
              if 'file_pos' in ac_state:
                  inp.seek(ac_state['file_pos'])
              else:
                   print("Warning: Input target stream position unverified! AC pointer lost.")
                   
              print(f"Restored continuous AC checkpoint state. Current buffer offset: {inp.tell()}")

          else:
              print("Warning: Critical AC parameters skipped. Resumption integrity unverified.")

      # Trigger main decompression sequence
      process(False, length, vocab_size, dec, output)
      
      # Set explicit read/write target modes accounting for fragmented file architectures.
      mode_write = "r+b" if (current_part > 1 and os.path.exists(output_path)) else "wb"
      
      with open(output_path, mode_write) as out:
        if preprocess == 'nncp' or preprocess == 'nncp-done':
            for i in range(length):
                out.write(bytes(((output[i] // 256),)))
                out.write(bytes(((output[i] % 256),)))
        else:
            # Map index translations properly tracking original character outputs.
            idx2char = np.array(vocab)
            for i in range(length):
                 out.write(bytes((idx2char[output[i]],)))

  if preprocess == 'cmix':
    if current_part == total_parts:
        !./cmix/cmix -d ./cmix/dictionary/english.dic $output_path ./data/final.dat
        output_path = "data/final.dat"
  if preprocess == 'nncp' or preprocess == 'nncp-done':
    if current_part == total_parts:
        !./nncp/preprocess d data/dictionary.words $output_path ./data/final.dat
        output_path = "data/final.dat"

In [ ]:
#@title Download Result

if mode == 'decompress':
  if preprocess == 'nncp-done':
    download('data/decompressed.dat')
  else:
    download(output_path)
  if checkpoint:
    import shutil
    shutil.make_archive('data/ckpt', 'zip', 'data/ckpt')
    download('data/ckpt.zip')

In [ ]:
#@title Validation

if mode == 'decompress' or mode == 'both':
  if preprocess == 'nncp-done':
    !md5sum data/decompressed.dat
  !md5sum $output_path
if mode == 'both':
  !md5sum $original_file

In [ ]:
#@title Close Runtime

import time
time.sleep(60)

from google.colab import runtime
runtime.unassign()